# PCA Ablation — Does Dimensionality Reduction Help Here?
### Dataset: CICIDS-2017  ·  Runs on **Kaggle**

## The question

A natural reviewer question is *"did you try PCA?"* This notebook answers it with evidence
rather than opinion. We compare models trained on the **47 selected features** against models
trained on **PCA-reduced** versions of the same features, on both tasks.

## Why we expect PCA *not* to help (the hypothesis we are testing)

1. **47 features is not a dimensionality problem** — tree models and a small MLP handle it
   trivially. PCA earns its keep at hundreds/thousands of features, not dozens.
2. **PCA optimises for variance, not class separation.** The highest-variance directions here
   are the heavy-tailed flow magnitudes, which are not necessarily the ones that separate
   attack from benign. PCA can discard a low-variance but highly discriminative feature.
3. **It destroys interpretability** — components are blends of all 47 originals, so we could no
   longer say "Init_Win is the rare-class signal".
4. **Tree models are rotation-sensitive in a good way** — they split on individual features;
   PCA's rotation of the space tends to make them slightly *worse*.

If the result confirms this — PCA equal-or-worse — that *justifies* keeping the
correlation-based feature selection instead of PCA. If PCA surprisingly *helps*, that is also
a worthwhile finding. Either way the experiment is informative.

**Leakage guard:** PCA is `fit` on the **training set only**, then applied to test — exactly
like the scaler. Fitting PCA on the full data before splitting would itself be a leak.

**Kaggle setup:** attach the FeatureSelection output dataset; set `IN_DIR`. No GPU needed.

## 1. Imports & Config

In [ ]:
import os, json, time, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi']=110; plt.rcParams['savefig.bbox']='tight'

RANDOM_SEED=42; np.random.seed(RANDOM_SEED)
CLASS_NAMES=['BENIGN','Bot/Infiltration','Brute Force','DDoS','DoS','PortScan','Web Attack']
N_CLASSES=len(CLASS_NAMES)

IN_DIR='/kaggle/input/ids-selected'   # EDIT to your FeatureSelection mount path
OUT_DIR='/kaggle/working'; FIGURES_DIR=os.path.join(OUT_DIR,'figures'); os.makedirs(FIGURES_DIR,exist_ok=True)

_report=[]
def _log(t=''): _report.append(str(t)); print(t)
def _savefig(n,fig=None): p=os.path.join(FIGURES_DIR,n); (fig or plt).savefig(p,dpi=130,bbox_inches='tight'); return p
def write_report():
    p=os.path.join(OUT_DIR,'Ablation_PCA_Report.txt')
    open(p,'w',encoding='utf-8').write('\n'.join(_report)); print('Report ->',p)

_log('='*70); _log('PCA ABLATION  —  CICIDS-2017')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}'); _log('='*70)
print('Setup complete.')

## 2. Load Data & Fit PCA on Training Set Only

In [ ]:
train_path=os.path.join(IN_DIR,'train_selected.parquet')
test_path =os.path.join(IN_DIR,'test_selected.parquet')
feat_path =os.path.join(IN_DIR,'selected_features.json')
for p in [train_path,test_path,feat_path]:
    if not os.path.exists(p): raise FileNotFoundError(f'{p} not found. Set IN_DIR.')
selected_features=json.load(open(feat_path))['selected_features']
train_df=pd.read_parquet(train_path); test_df=pd.read_parquet(test_path)
n_features=len(selected_features)

# subsample train for speed (test stays full for honest metrics)
SAMPLE_N=300_000
tr=(train_df.groupby('label_multi',group_keys=False)
      .apply(lambda g:g.sample(min(len(g),max(1,SAMPLE_N*len(g)//len(train_df))),random_state=RANDOM_SEED)))

X_tr=tr[selected_features].values.astype(np.float32)
X_te=test_df[selected_features].values.astype(np.float32)

# PCA fit on TRAIN only (features are already StandardScaled from feature engineering)
pca_full=PCA(n_components=n_features, random_state=RANDOM_SEED).fit(X_tr)
cum_var=np.cumsum(pca_full.explained_variance_ratio_)
n_95=int(np.argmax(cum_var>=0.95)+1)
n_99=int(np.argmax(cum_var>=0.99)+1)

_log(''); _log('── SECTION 2 : DATA + PCA FIT (train only) ────────────────')
_log(f'  Features (full)        : {n_features}')
_log(f'  Train subsample        : {len(tr):,}   Test (full) : {len(test_df):,}')
_log(f'  Components for 95% var : {n_95}')
_log(f'  Components for 99% var : {n_99}')
_log(f'  -> PCA needs {n_95}/{n_features} components to keep 95% of the variance, i.e. only '
     f'{n_features-n_95} fewer dims. Modest compression on an already-small feature set.')

# scree / cumulative-variance plot
fig,ax=plt.subplots(figsize=(9,4.5))
ax.plot(range(1,n_features+1),cum_var,'o-',color='#1976D2')
ax.axhline(0.95,ls='--',color='green',label='95% variance')
ax.axhline(0.99,ls=':',color='orange',label='99% variance')
ax.axvline(n_95,ls='--',color='green',alpha=0.4)
ax.set_xlabel('number of principal components'); ax.set_ylabel('cumulative explained variance')
ax.set_title('PCA cumulative variance (fit on training set)',fontweight='bold'); ax.legend()
plt.tight_layout(); _savefig('01_pca_variance.png',fig); plt.show()

## 3. F1 vs Number of Components
Train the same Random Forest on PCA projections of increasing size, and on the full feature
set, for both tasks. The question: does any PCA setting match or beat the full 47 features?

In [ ]:
def fit_eval(Xtr, Xte, task):
    ycol='label_binary' if task=='binary' else 'label_multi'
    ytr=tr[ycol].values; yte=test_df[ycol].values
    clf=RandomForestClassifier(n_estimators=120,n_jobs=-1,class_weight='balanced',random_state=RANDOM_SEED)
    clf.fit(Xtr,ytr); pred=clf.predict(Xte)
    if task=='binary': return f1_score(yte,pred), accuracy_score(yte,pred)
    return f1_score(yte,pred,average='macro'), accuracy_score(yte,pred)

comp_grid=sorted(set([2,5,10,15,20,30,n_95,n_features]))
comp_grid=[c for c in comp_grid if c<=n_features]

_log(''); _log('── SECTION 3 : F1 vs #COMPONENTS ──────────────────────────')
rows=[]
for k in comp_grid:
    t0=time.time()
    if k==n_features:
        Xtr_k, Xte_k, tag = X_tr, X_te, 'FULL (no PCA)'
    else:
        pca=PCA(n_components=k, random_state=RANDOM_SEED).fit(X_tr)
        Xtr_k, Xte_k, tag = pca.transform(X_tr), pca.transform(X_te), f'PCA-{k}'
    bf1,bacc=fit_eval(Xtr_k,Xte_k,'binary')
    mf1,macc=fit_eval(Xtr_k,Xte_k,'multi')
    rows.append({'components':k,'tag':tag,'binary_f1':bf1,'multi_macro_f1':mf1,
                 'binary_acc':bacc,'multi_acc':macc})
    _log(f'  {tag:<14} ({k:>2} dims) | binary F1={bf1:.4f}  multi macroF1={mf1:.4f}  ({time.time()-t0:.1f}s)')

res=pd.DataFrame(rows)
full_bin =res.loc[res.tag=='FULL (no PCA)','binary_f1'].values[0]
full_mult=res.loc[res.tag=='FULL (no PCA)','multi_macro_f1'].values[0]
display(res)

## 4. Results & Verdict

In [ ]:
pca_rows=res[res.tag!='FULL (no PCA)']
best_pca_mult=pca_rows.loc[pca_rows['multi_macro_f1'].idxmax()]
best_pca_bin =pca_rows.loc[pca_rows['binary_f1'].idxmax()]

_log(''); _log('='*70); _log('SECTION 4 : VERDICT'); _log('='*70)
_log(f'  FULL 47 features        : binary F1={full_bin:.4f}   multi macroF1={full_mult:.4f}')
_log(f'  Best PCA (binary)       : {best_pca_bin.tag} F1={best_pca_bin.binary_f1:.4f}  '
     f'(delta {best_pca_bin.binary_f1-full_bin:+.4f})')
_log(f'  Best PCA (multi-class)  : {best_pca_mult.tag} macroF1={best_pca_mult.multi_macro_f1:.4f}  '
     f'(delta {best_pca_mult.multi_macro_f1-full_mult:+.4f})')

delta_mult=best_pca_mult.multi_macro_f1-full_mult
if delta_mult>0.005:
    v='PCA HELPED — unexpected; worth keeping. Re-examine feature selection.'
elif delta_mult>-0.01:
    v='PCA is ~EQUAL to the full feature set, but costs all interpretability and adds a transform step. Keep the selected features.'
else:
    v='PCA HURT performance (esp. macro-F1 / rare classes), as hypothesised. Keep the selected features.'
_log(''); _log('  '+v); print('\nVerdict:',v)

# plot F1 vs components
fig,ax=plt.subplots(figsize=(10,5))
pr=pca_rows.sort_values('components')
ax.plot(pr['components'],pr['binary_f1'],'o-',label='PCA binary F1',color='#1976D2')
ax.plot(pr['components'],pr['multi_macro_f1'],'s-',label='PCA multi macro-F1',color='#7B1FA2')
ax.axhline(full_bin,ls='--',color='#1976D2',alpha=0.6,label=f'FULL binary F1 = {full_bin:.3f}')
ax.axhline(full_mult,ls='--',color='#7B1FA2',alpha=0.6,label=f'FULL multi macro-F1 = {full_mult:.3f}')
ax.set_xlabel('number of PCA components'); ax.set_ylabel('test F1'); ax.set_ylim(0,1.02)
ax.set_title('PCA components vs Full feature set — does PCA ever catch up?',fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout(); _savefig('02_pca_vs_full.png',fig); plt.show()

res.to_csv(os.path.join(OUT_DIR,'pca_ablation.csv'),index=False)
write_report(); print('Saved pca_ablation.csv + report.')